# Data Imports

In [9]:
import pandas as pd
import numpy as np
import json
import math

In [10]:
# import Artists dataset from Getty, csv file downloaded 10/22/26
Artists = pd.read_csv("Artists.csv")
Artists.head(5)

,x,name,bio,nationality,type
0,http://vocab.getty.edu/ulan/500771934,"Aa, Andries van der","North Netherlandish painter, 1743-1776",North Netherlandish,painters (artists)
1,http://vocab.getty.edu/ulan/500557119,"Aabdaoui, Latif","Moroccan graphic artist, 1958-",Moroccan,graphic artists
2,http://vocab.getty.edu/ulan/500632715,"Aa, Ben","Dutch painter, born 1951",Dutch (culture or style),painters (artists)
3,http://vocab.getty.edu/ulan/500546545,"Aaberg, Gunhild","Danish ceramicist, 1939-",Danish (culture or style),ceramicists
4,http://vocab.getty.edu/ulan/500108834,"Aabergh, Gösta","Norwegian architect, active mid-late 20th century",Norwegian (culture),architects


In [11]:
# Import Art Institute of Chicago dataset, jsonl file from 10/22/26
Chicago = pd.read_json("AIofChicago.jsonl", lines = True) 
Chicago.head(5)

,id,title,main_reference_number,department_title,artist_title
0,4,Priest and Boy,1880.1,Prints and Drawings,Lawrence Carmichael Earle
1,9,"Interior of St. Mark's, Venice",1887.232,Arts of the Americas,David Dalhoff Neal
2,11,Self-Portrait,1887.234,Arts of the Americas,Walter Shirlaw
3,16,The Fall of the Giants,1887.249,Prints and Drawings,Salvator Rosa
4,19,"View of Ponte Lugano on the Anio, from Views o...",1887.252,Prints and Drawings,Giovanni Battista Piranesi


In [44]:
Chicago[Chicago["main_reference_number"].str.split(".", expand = True)[0].str.isnumeric() == False]

,id,title,main_reference_number,department_title,artist_title
3343,6605,Figure Asleep on Mashrabiya Background,Obj: 6605,Prints and Drawings,Henri Matisse
19824,31978,Colophon from Poèmes et Lithographies,RX7243/1,Prints and Drawings,Pablo Picasso
21770,35442,Ceres,Obj: 35442,Prints and Drawings,Francesco Bartolozzi
24024,39913,"Title Page for the ""Odes and Sonnets of John K...",Obj: 39913,Prints and Drawings,Will Hicock Low
32121,52356,A Bullfight or Luisa's Love (Una Corrida de To...,Obj: 52356,Prints and Drawings,José Guadalupe Posada
...,...,...,...,...,...
139259,283218,4 Pentagons,Obj: 283218,Prints and Drawings,Robert Mangold
139260,283219,4 Pentagons,Obj: 283219,Prints and Drawings,Robert Mangold
139261,283220,4 Pentagons,Obj: 283220,Prints and Drawings,Robert Mangold
139262,283221,4 Pentagons,Obj: 283221,Prints and Drawings,Robert Mangold


In [23]:
# Import The Met's dataset from their github
Met = pd.read_csv("Cleaned-Met-Objects.csv", keep_default_na = False, na_values = [""]) 
Met.head(5)

,Object Number,Object ID,Department,Original_AccessionYear,AY_Clean,Object Name,Title,Clean_Title,Artist Display Name
0,1979.486.1,1,The American Wing,1979.0,1979.0,Coin,One-dollar Liberty Head Coin,One-dollar Liberty Head Coin,James Barton Longacre
1,1980.264.5,2,The American Wing,1980.0,1980.0,Coin,Ten-dollar Liberty Head Coin,Ten-dollar Liberty Head Coin,Christian Gobrecht
2,67.265.9,3,The American Wing,1967.0,1967.0,Coin,Two-and-a-Half Dollar Coin,Two-and-a-Half Dollar Coin,NaN
3,67.265.10,4,The American Wing,1967.0,1967.0,Coin,Two-and-a-Half Dollar Coin,Two-and-a-Half Dollar Coin,NaN
4,67.265.11,5,The American Wing,1967.0,1967.0,Coin,Two-and-a-Half Dollar Coin,Two-and-a-Half Dollar Coin,NaN


In [25]:
Met.isna().sum()

Object Number                  0
Object ID                      0
Department                     0
Original_AccessionYear      4169
AY_Clean                    3395
Object Name                 2587
Title                      27776
Clean_Title                    3
Artist Display Name       199720
dtype: int64

In [28]:
Met = Met.drop(columns = ["Original_AccessionYear", "Object Name", "Title"])

The data we imported into OpenRefine comes from the Met's github, at https://github.com/metmuseum/openaccess/blob/master/MetObjects.csv, which we pared down to a subset, and then cleaned. The subset we are focusing on has variables `Object Number`, `Object ID`, `Department`, `Origial_AccessionYear`, `AY_Clean`, `Object Name`, `Title`, `Clean_Title`, and `Artist Display Name` chosen to match the Art Institute of Chicago dataset. 

The two versions of Accession Year are a pre-cleaning version, which retains the information from the original dataset, and the cleaned version: `AY_Clean`, which supplements the information with information derived from `Object Number`, and a (now deleted) column Credit Line, which sometimes included the year a piece was aquired. `Clean_Title` is largely a combination of `Title` and `Object Name`, with `Title` taking priority. There are still missing values in the dataset, but the cleaned columns reduce them down. We drop the original columns from here on out. 

___

## Record Linkage

In [19]:
Chicago.head()

,id,title,main_reference_number,department_title,artist_title
0,4,Priest and Boy,1880.1,Prints and Drawings,Lawrence Carmichael Earle
1,9,"Interior of St. Mark's, Venice",1887.232,Arts of the Americas,David Dalhoff Neal
2,11,Self-Portrait,1887.234,Arts of the Americas,Walter Shirlaw
3,16,The Fall of the Giants,1887.249,Prints and Drawings,Salvator Rosa
4,19,"View of Ponte Lugano on the Anio, from Views o...",1887.252,Prints and Drawings,Giovanni Battista Piranesi


In [21]:
Met.head()

,Object Number,Department,AccessionYear,Object Name,Title,Artist Display Name,Cleaned_AY,AYfromON
0,1979.486.1,The American Wing,1979.0,Coin,One-dollar Liberty Head Coin,James Barton Longacre,1979.0,1979
1,1980.264.5,The American Wing,1980.0,Coin,Ten-dollar Liberty Head Coin,Christian Gobrecht,1980.0,1980
2,67.265.9,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0,67
3,67.265.10,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0,67
4,67.265.11,The American Wing,1967.0,Coin,Two-and-a-Half Dollar Coin,NaN,1967.0,67


In [22]:
Artists.head()

,x,name,bio,nationality,type
0,http://vocab.getty.edu/ulan/500771934,"Aa, Andries van der","North Netherlandish painter, 1743-1776",North Netherlandish,painters (artists)
1,http://vocab.getty.edu/ulan/500557119,"Aabdaoui, Latif","Moroccan graphic artist, 1958-",Moroccan,graphic artists
2,http://vocab.getty.edu/ulan/500632715,"Aa, Ben","Dutch painter, born 1951",Dutch (culture or style),painters (artists)
3,http://vocab.getty.edu/ulan/500546545,"Aaberg, Gunhild","Danish ceramicist, 1939-",Danish (culture or style),ceramicists
4,http://vocab.getty.edu/ulan/500108834,"Aabergh, Gösta","Norwegian architect, active mid-late 20th century",Norwegian (culture),architects


In [20]:
import recordlinkage as rl

In [21]:
# preprocessing

# standardize artist names across all three datasets
Artists["clean_name"] = (Artists["name"].str.replace(r"^([^,]+),\s*(.+)$", r"\2 \1", regex=True).str.lower().str.strip())
Chicago["clean_artist"] = Chicago["artist_title"].str.lower().str.strip()
Met["clean_artist"] = Met["Artist Display Name"].str.lower().str.strip()

artists_clean = (
    Artists[["x", "name", "clean_name", "bio", "nationality", "type"]]
    .dropna(subset=["clean_name"])
    .drop_duplicates(subset=["clean_name"])
    .reset_index(drop=True)
)

chicago_artists = (
    Chicago[["clean_artist"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

met_artists = (
    Met[["clean_artist"]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

artists_clean["block_key"] = artists_clean["clean_name"].str[:3]
chicago_artists["block_key"] = chicago_artists["clean_artist"].str[:3]
met_artists["block_key"] = met_artists["clean_artist"].str[:3]

In [22]:
def link_datasets(museum_df, getty_df, threshold=0.85):
    indexer = rl.Index()
    indexer.block("block_key")
    pairs = indexer.index(museum_df, getty_df)
    
    compare = rl.Compare()
    compare.string("clean_artist", "clean_name", method="levenshtein", threshold=threshold, label="name_sim")
    features = compare.compute(pairs, museum_df, getty_df)
    
    candidates = features[features.sum(axis=1) == 1]
    return candidates

In [23]:
def get_matched_pairs(candidates, museum_df, getty_df):
    match_pairs = candidates.index.to_frame(index=False)
    match_pairs.columns = ["museum_index", "getty_index"]
    
    merged = (
        match_pairs
        .merge(museum_df, left_on="museum_index", right_index=True)
        .merge(getty_df, left_on="getty_index", right_index=True, suffixes=("_museum", "_getty"))
    )
    return merged

In [24]:
chicago_candidates = link_datasets(chicago_artists, artists_clean, threshold=0.85)
chicago_matched_pairs_df = get_matched_pairs(chicago_candidates, chicago_artists, artists_clean)

In [25]:
met_candidates = link_datasets(met_artists, artists_clean, threshold=0.85)
met_matched_pairs_df = get_matched_pairs(met_candidates, met_artists, artists_clean)

In [26]:
# record fusion

def fuse_records(matched_df):
    fused_df = pd.DataFrame()
    
    if "clean_artist" in matched_df.columns:
        fused_df["clean_artist"] = matched_df["clean_artist"]
    elif "clean_artist_museum" in matched_df.columns:
        fused_df["clean_artist"] = matched_df["clean_artist_museum"]
        
    if "name" in matched_df.columns:
        fused_df["artist_name"] = matched_df["name"]
    elif "name_getty" in matched_df.columns:
        fused_df["artist_name"] = matched_df["name_getty"]
        
    if "x" in matched_df.columns:
        fused_df["getty_uri"] = matched_df["x"]
    elif "x_getty" in matched_df.columns:
        fused_df["getty_uri"] = matched_df["x_getty"]
        
    if "bio" in matched_df.columns:
        fused_df["artist_bio"] = matched_df["bio"]
    elif "bio_getty" in matched_df.columns:
        fused_df["artist_bio"] = matched_df["bio_getty"]
        
    if "nationality" in matched_df.columns:
        fused_df["artist_nationality"] = matched_df["nationality"]
    elif "nationality_getty" in matched_df.columns:
        fused_df["artist_nationality"] = matched_df["nationality_getty"]
        
    return fused_df

In [27]:
chicago_fused_df = fuse_records(chicago_matched_pairs_df)
chicago_fused_df.head()

,clean_artist,artist_name,getty_uri,artist_bio,artist_nationality
0,lawrence carmichael earle,"Earle, Lawrence Carmichael",http://vocab.getty.edu/ulan/500029948,"American painter, 1845-1921",American (North American)
1,david dalhoff neal,"Neal, David Dalhoff",http://vocab.getty.edu/ulan/500026278,"American painter, 1838-1915",American (North American)
2,walter shirlaw,"Shirlaw, Walter",http://vocab.getty.edu/ulan/500002781,"American painter and illustrator, 1838-1909",American (North American)
3,salvator rosa,"Rosa, Salvator",http://vocab.getty.edu/ulan/500011328,"Italian painter and engraver, 1615-1673",Italian (culture or style)
4,giovanni battista piranesi,"Piranesi, Giovanni Battista",http://vocab.getty.edu/ulan/500114965,"Italian printmaker, architect, 1720-1778",Italian (culture or style)


In [28]:
met_fused_df = fuse_records(met_matched_pairs_df)
met_fused_df.head()

,clean_artist,artist_name,getty_uri,artist_bio,artist_nationality
0,james barton longacre,"Longacre, James Barton",http://vocab.getty.edu/ulan/500011409,"American printmaker and painter, 1794-1869",American (North American)
1,christian gobrecht,"Gobrecht, Christian",http://vocab.getty.edu/ulan/500077295,"American engraver, 1785-1844",American (North American)
2,christian gobrecht,"Robbrecht, Christian",http://vocab.getty.edu/ulan/500213310,"Belgian architect, contemporary",Belgian (modern)
3,bela lyon pratt,"Pratt, Bela Lyon",http://vocab.getty.edu/ulan/500095555,"American sculptor, 1867-1917",American (North American)
4,pierre stephan,"Stephan, Pierre",http://vocab.getty.edu/ulan/500100483,"French ceramicist, active 1770-1795, active in...",French (culture or style)


In [30]:
chicago_fused_df.to_csv('chicago_fused.csv', index=False)
met_fused_df.to_csv('met_fused.csv', index=False)